# Decay analyse + OrcaFlex simulatie zonder demping

Dit notebook leest decay `.h5m` experimenten in, detecteert A1/A2, start de simulatie op A2 en runt OrcaFlex met lineaire en kwadratische demping op nul. Daarna worden periode, overlay-plots en een Excel-overzicht opgeslagen.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from scipy.signal import find_peaks

try:
    import OrcFxAPI
    ORCAFLEX_AVAILABLE = True
except ImportError:
    ORCAFLEX_AVAILABLE = False
    warnings.warn("OrcFxAPI niet beschikbaar: signaalanalyse werkt wel, simulatie niet.")

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

In [2]:
# =========================
# CONFIGURATIE - PAS AAN
# =========================
RAW_DATA_ROOT = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\Decay_Data\01_Rawdata")
OUTPUT_DIR = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\Decay_no_damping_analysis")
PLOTS_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DOF = "pitch"  # "pitch", "roll" of "heave"
SIGNAL_FOR_ANALYSIS = "unfiltered"  # "unfiltered" of "filtered"
MIN_ABS_INITIAL_AMPLITUDE = 2.0
MIN_PEAK_DISTANCE_S = 2.0
SIM_DURATION_AFTER_A2 = 100.0
PLOT_START_BEFORE_A1 = 5.0

MODEL_BY_CONSTRUCTION = {
    "fixedwith": Path(r"C:\path\to\fixedwith_model.dat"),
    "fixedwithout": Path(r"C:\path\to\fixedwithout_model.dat"),
    "floating": Path(r"C:\path\to\floating_model.dat"),
}
FLOATER_OBJECT_NAME = "floaters"

CONSTRUCTION_KEYWORDS = {
    "fixedwith": ["fixedwith", "fixed_with", "withdeck", "with_deck"],
    "fixedwithout": ["fixedwithout", "fixed_without", "withoutdeck", "without_deck"],
    "floating": ["floating", "float"],
}

In [3]:
def read_decay_signals(exp_path: Path, dof: str):
    filtered_map = {
        "pitch": "CroppedSignals/PITCH (LPF: 5.0 rad*s^-1)",
        "roll": "CroppedSignals/ROLL (LPF: 5.0 rad*s^-1)",
        "heave": "CroppedSignals/Z_COG (LPF: 5.0 rad*s^-1)",
    }
    unfiltered_map = {
        "pitch": "UnfilteredSignals/PITCH (unfiltered)",
        "roll": "UnfilteredSignals/ROLL (unfiltered)",
        "heave": "UnfilteredSignals/Z_COG (unfiltered)",
    }
    with h5py.File(exp_path, "r") as f:
        return {
            "filtered": (np.asarray(f["CroppedSignals/time"]), np.asarray(f[filtered_map[dof]])),
            "unfiltered": (np.asarray(f["UnfilteredSignals/time"]), np.asarray(f[unfiltered_map[dof]])),
        }

def find_experiment_files(root: Path):
    return sorted([p for p in root.rglob("*.h5m") if "decay" in p.name.lower()])

def infer_construction(exp_path: Path):
    text = str(exp_path).lower()
    for construction, keys in CONSTRUCTION_KEYWORDS.items():
        if any(k in text for k in keys):
            return construction
    return "unknown"

experiment_files = find_experiment_files(RAW_DATA_ROOT)
print(f"Aantal decaybestanden gevonden: {len(experiment_files)}")
experiment_files[:5]

Aantal decaybestanden gevonden: 49


[WindowsPath('C:/Users/verav/Desktop/Studie/Afstuderen/Decay_Data/01_Rawdata/02/002/34224_03CB_02_002_001_01_Decay1.h5m'),
 WindowsPath('C:/Users/verav/Desktop/Studie/Afstuderen/Decay_Data/01_Rawdata/02/002/34224_03CB_02_002_001_01_Decay2.h5m'),
 WindowsPath('C:/Users/verav/Desktop/Studie/Afstuderen/Decay_Data/01_Rawdata/02/002/34224_03CB_02_002_001_01_Decay3.h5m'),
 WindowsPath('C:/Users/verav/Desktop/Studie/Afstuderen/Decay_Data/01_Rawdata/02/002/34224_03CB_02_002_002_01_Decay1.h5m'),
 WindowsPath('C:/Users/verav/Desktop/Studie/Afstuderen/Decay_Data/01_Rawdata/02/002/34224_03CB_02_002_003_01_Decay1.h5m')]

In [4]:
def _min_peak_distance_samples(t, min_distance_s):
    dt = np.nanmedian(np.diff(t)) if len(t) > 1 else np.nan
    if not np.isfinite(dt) or dt <= 0:
        return 1
    return max(1, int(round(min_distance_s / dt)))

def detect_peaks_and_troughs(t, z, min_distance_s=2.0):
    distance = _min_peak_distance_samples(t, min_distance_s)
    peaks, _ = find_peaks(z, distance=distance)
    troughs, _ = find_peaks(-z, distance=distance)
    extrema_idx = np.sort(np.r_[peaks, troughs])
    return peaks, troughs, extrema_idx

def select_A1_A2(t, z, extrema_idx, min_abs_amp=2.0):
    valid = [i for i in extrema_idx if abs(z[i]) >= min_abs_amp]
    if len(valid) < 2:
        raise ValueError(f"Minder dan 2 extrema met |amplitude| >= {min_abs_amp}")
    i1, i2 = valid[0], valid[1]
    return {"A1_idx": i1, "A1_t": float(t[i1]), "A1_z": float(z[i1]),
            "A2_idx": i2, "A2_t": float(t[i2]), "A2_z": float(z[i2])}

def estimate_period_from_extrema(t, peaks, troughs):
    periods = []
    if len(peaks) >= 2:
        periods.extend(np.diff(t[peaks]))
    if len(troughs) >= 2:
        periods.extend(np.diff(t[troughs]))
    periods = np.asarray(periods, dtype=float)
    periods = periods[np.isfinite(periods)]
    if len(periods) == 0:
        return np.nan, np.nan, 0
    return float(np.mean(periods)), float(np.std(periods)), int(len(periods))

In [5]:
def set_no_damping(floater, dof: str):
    # Pas deze propertynamen aan als jouw OrcaFlex-model andere namen gebruikt.
    attr_map = {
        "heave": ["OtherDampingLinearZ", "OtherDampingQuadraticZ", "LinearDampingZ", "QuadraticDampingZ"],
        "pitch": ["OtherDampingLinearRy", "OtherDampingQuadraticRy", "LinearDampingRy", "QuadraticDampingRy",
                  "OtherDampingLinearPitch", "OtherDampingQuadraticPitch"],
        "roll": ["OtherDampingLinearRx", "OtherDampingQuadraticRx", "LinearDampingRx", "QuadraticDampingRx",
                 "OtherDampingLinearRoll", "OtherDampingQuadraticRoll"],
    }
    changed = []
    for attr in attr_map[dof]:
        if hasattr(floater, attr):
            try:
                setattr(floater, attr, 0.0)
                changed.append(attr)
            except Exception:
                pass
    return changed

def set_initial_amplitude(floater, dof: str, amplitude: float):
    for attr in ["InitialZ", "InitialX", "InitialY", "InitialHeel", "InitialTrim", "InitialHeading"]:
        if hasattr(floater, attr):
            try: setattr(floater, attr, 0.0)
            except Exception: pass
    if dof == "heave": floater.InitialZ = amplitude
    if dof == "roll": floater.InitialHeel = amplitude
    if dof == "pitch": floater.InitialTrim = amplitude

def get_simulated_time_history(floater, dof: str):
    var_name = {"heave": "Z", "roll": "Rotation 1", "pitch": "Rotation 2"}[dof]
    return np.asarray(floater.TimeHistory("Time")), np.asarray(floater.TimeHistory(var_name))

def run_orcaflex_no_damping(model_path: Path, dof: str, initial_amplitude: float, duration: float):
    if not ORCAFLEX_AVAILABLE:
        raise RuntimeError("OrcFxAPI is niet beschikbaar.")
    if not model_path.exists():
        raise FileNotFoundError(f"Modelbestand bestaat niet: {model_path}")
    model = OrcFxAPI.Model(str(model_path))
    floater = model[FLOATER_OBJECT_NAME]
    changed = set_no_damping(floater, dof)
    set_initial_amplitude(floater, dof, initial_amplitude)
    try:
        model.general.StageDuration = [duration]
    except Exception:
        warnings.warn("StageDuration kon niet worden gezet; controleer je OrcaFlex stages.")
    model.RunSimulation()
    t_sim, z_sim = get_simulated_time_history(floater, dof)
    return t_sim, z_sim, changed

In [6]:
results = []
for exp_path in experiment_files:
    try:
        construction = infer_construction(exp_path)
        t_exp, z_exp = read_decay_signals(exp_path, DOF)[SIGNAL_FOR_ANALYSIS]
        peaks, troughs, extrema_idx = detect_peaks_and_troughs(t_exp, z_exp, MIN_PEAK_DISTANCE_S)
        a = select_A1_A2(t_exp, z_exp, extrema_idx, MIN_ABS_INITIAL_AMPLITUDE)
        T_exp_mean, T_exp_std, T_exp_n = estimate_period_from_extrema(t_exp, peaks, troughs)

        t_sim, z_sim = np.array([]), np.array([])
        T_sim_mean, T_sim_std, T_sim_n = np.nan, np.nan, 0
        changed_damping_attrs, error = [], ""
        sim_status = "not_run"
        model_path = MODEL_BY_CONSTRUCTION.get(construction)

        if model_path is None:
            error = f"Geen modelpad voor construction={construction}"
        elif not ORCAFLEX_AVAILABLE:
            error = "OrcFxAPI niet beschikbaar"
        else:
            try:
                t_sim, z_sim, changed_damping_attrs = run_orcaflex_no_damping(model_path, DOF, a["A2_z"], SIM_DURATION_AFTER_A2)
                sim_peaks, sim_troughs, _ = detect_peaks_and_troughs(t_sim, z_sim, MIN_PEAK_DISTANCE_S)
                T_sim_mean, T_sim_std, T_sim_n = estimate_period_from_extrema(t_sim, sim_peaks, sim_troughs)
                sim_status = "ok"
            except Exception as e:
                sim_status = "failed"
                error = str(e)

        plot_start = a["A1_t"] - PLOT_START_BEFORE_A1
        plot_end = a["A2_t"] + SIM_DURATION_AFTER_A2
        mask = (t_exp >= plot_start) & (t_exp <= plot_end)
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(t_exp[mask], z_exp[mask], label="Experiment")
        ax.scatter([a["A1_t"], a["A2_t"]], [a["A1_z"], a["A2_z"]], marker="x", s=90, label="A1/A2")
        if len(t_sim) > 0:
            ax.plot(t_sim + a["A2_t"], z_sim, label="Simulatie zonder demping")
        ax.axvline(a["A2_t"], linestyle="--", linewidth=1, label="Start simulatie A2")
        ax.set_xlabel("tijd [s]")
        ax.set_ylabel(DOF)
        ax.set_title(f"{exp_path.name} | {construction} | T_exp={T_exp_mean:.3f}s | T_sim={T_sim_mean:.3f}s | {sim_status}")
        ax.legend()
        fig.tight_layout()
        plot_path = PLOTS_DIR / f"{DOF}_{construction}_{exp_path.stem}_no_damping.png"
        fig.savefig(plot_path, dpi=200)
        plt.close(fig)

        results.append({
            "experiment": exp_path.name, "exp_path": str(exp_path), "construction": construction, "dof": DOF,
            "A1_t": a["A1_t"], "A1_amp": a["A1_z"], "A2_t": a["A2_t"], "A2_amp": a["A2_z"],
            "T_exp_mean": T_exp_mean, "T_exp_std": T_exp_std, "T_exp_n": T_exp_n,
            "T_sim_mean": T_sim_mean, "T_sim_std": T_sim_std, "T_sim_n": T_sim_n,
            "period_difference_abs": T_sim_mean - T_exp_mean if np.isfinite(T_sim_mean) and np.isfinite(T_exp_mean) else np.nan,
            "period_difference_pct": 100*(T_sim_mean - T_exp_mean)/T_exp_mean if np.isfinite(T_sim_mean) and np.isfinite(T_exp_mean) and T_exp_mean != 0 else np.nan,
            "sim_status": sim_status, "changed_damping_attrs": ", ".join(changed_damping_attrs), "plot_path": str(plot_path), "error": error,
        })
    except Exception as e:
        warnings.warn(f"Fout bij {exp_path}: {e}")
        results.append({"experiment": exp_path.name, "exp_path": str(exp_path), "construction": infer_construction(exp_path), "dof": DOF, "sim_status": "analysis_failed", "error": str(e)})

results_df = pd.DataFrame(results)
results_df.head()

C:\Users\verav\AppData\Local\Temp\ipykernel_16284\524869789.py:58: UserWarning: Fout bij C:\Users\verav\Desktop\Studie\Afstuderen\Decay_Data\01_Rawdata\02\002\34224_03CB_02_002_003_01_Decay1.h5m: "Unable to synchronously open object (object 'PITCH (LPF: 5.0 rad*s^-1)' doesn't exist)"
  warnings.warn(f"Fout bij {exp_path}: {e}")
C:\Users\verav\AppData\Local\Temp\ipykernel_16284\524869789.py:58: UserWarning: Fout bij C:\Users\verav\Desktop\Studie\Afstuderen\Decay_Data\01_Rawdata\02\002\34224_03CB_02_002_003_01_Decay2.h5m: "Unable to synchronously open object (object 'PITCH (LPF: 5.0 rad*s^-1)' doesn't exist)"
  warnings.warn(f"Fout bij {exp_path}: {e}")
C:\Users\verav\AppData\Local\Temp\ipykernel_16284\524869789.py:58: UserWarning: Fout bij C:\Users\verav\Desktop\Studie\Afstuderen\Decay_Data\01_Rawdata\02\002\34224_03CB_02_002_004_01_Decay1.h5m: "Unable to synchronously open object (object 'PITCH (LPF: 5.0 rad*s^-1)' doesn't exist)"
  warnings.warn(f"Fout bij {exp_path}: {e}")
C:\Users\v

,experiment,exp_path,construction,dof,A1_t,A1_amp,A2_t,A2_amp,T_exp_mean,T_exp_std,T_exp_n,T_sim_mean,T_sim_std,T_sim_n,period_difference_abs,period_difference_pct,sim_status,changed_damping_attrs,plot_path,error
0,34224_03CB_02_002_001_01_Decay1.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,351.693029,5.636448,355.367908,-6.104080,7.023089,0.961767,762.0,NaN,NaN,0.0,NaN,NaN,not_run,,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,Geen modelpad voor construction=unknown
1,34224_03CB_02_002_001_01_Decay2.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,351.693029,5.636448,355.367908,-6.104080,7.023089,0.961767,762.0,NaN,NaN,0.0,NaN,NaN,not_run,,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,Geen modelpad voor construction=unknown
2,34224_03CB_02_002_001_01_Decay3.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,351.693029,5.636448,355.367908,-6.104080,7.023089,0.961767,762.0,NaN,NaN,0.0,NaN,NaN,not_run,,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,Geen modelpad voor construction=unknown
3,34224_03CB_02_002_002_01_Decay1.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,3314.261174,7.669963,3317.936483,-8.120571,6.867592,1.660951,507.0,NaN,NaN,0.0,NaN,NaN,not_run,,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,Geen modelpad voor construction=unknown
4,34224_03CB_02_002_003_01_Decay1.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,analysis_failed,NaN,NaN,"""Unable to synchronously open object (object '..."


In [7]:
excel_out = OUTPUT_DIR / f"decay_periods_no_damping_{DOF}.xlsx"
with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="results", index=False)
    if len(results_df) > 0:
        summary = (results_df.groupby(["dof", "construction"], dropna=False)
                   .agg(n_experiments=("experiment", "count"), mean_T_exp=("T_exp_mean", "mean"),
                        std_T_exp=("T_exp_mean", "std"), mean_T_sim=("T_sim_mean", "mean"),
                        std_T_sim=("T_sim_mean", "std"), mean_diff_abs=("period_difference_abs", "mean"),
                        mean_diff_pct=("period_difference_pct", "mean"))
                   .reset_index())
        summary.to_excel(writer, sheet_name="summary", index=False)
print(f"Excel opgeslagen naar: {excel_out}")
print(f"Plots opgeslagen in: {PLOTS_DIR}")
results_df

Excel opgeslagen naar: C:\Users\verav\Desktop\Studie\Afstuderen\Decay_no_damping_analysis\decay_periods_no_damping_pitch.xlsx
Plots opgeslagen in: C:\Users\verav\Desktop\Studie\Afstuderen\Decay_no_damping_analysis\plots


,experiment,exp_path,construction,dof,A1_t,A1_amp,A2_t,A2_amp,T_exp_mean,T_exp_std,T_exp_n,T_sim_mean,T_sim_std,T_sim_n,period_difference_abs,period_difference_pct,sim_status,changed_damping_attrs,plot_path,error
0,34224_03CB_02_002_001_01_Decay1.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,351.693029,5.636448,355.367908,-6.104080,7.023089,0.961767,762.0,NaN,NaN,0.0,NaN,NaN,not_run,,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,Geen modelpad voor construction=unknown
1,34224_03CB_02_002_001_01_Decay2.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,351.693029,5.636448,355.367908,-6.104080,7.023089,0.961767,762.0,NaN,NaN,0.0,NaN,NaN,not_run,,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,Geen modelpad voor construction=unknown
2,34224_03CB_02_002_001_01_Decay3.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,351.693029,5.636448,355.367908,-6.104080,7.023089,0.961767,762.0,NaN,NaN,0.0,NaN,NaN,not_run,,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,Geen modelpad voor construction=unknown
3,34224_03CB_02_002_002_01_Decay1.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,3314.261174,7.669963,3317.936483,-8.120571,6.867592,1.660951,507.0,NaN,NaN,0.0,NaN,NaN,not_run,,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,Geen modelpad voor construction=unknown
4,34224_03CB_02_002_003_01_Decay1.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,analysis_failed,NaN,NaN,"""Unable to synchronously open object (object '..."
5,34224_03CB_02_002_003_01_Decay2.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,analysis_failed,NaN,NaN,"""Unable to synchronously open object (object '..."
6,34224_03CB_02_002_004_01_Decay1.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,analysis_failed,NaN,NaN,"""Unable to synchronously open object (object '..."
7,34224_03CB_02_002_004_01_Decay2.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,analysis_failed,NaN,NaN,"""Unable to synchronously open object (object '..."
8,34224_03CB_02_002_005_01_Decay1.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,analysis_failed,NaN,NaN,"""Unable to synchronously open object (object '..."
9,34224_03CB_02_002_006_01_Decay1.h5m,C:\Users\verav\Desktop\Studie\Afstuderen\Decay...,unknown,pitch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,analysis_failed,NaN,NaN,"""Unable to synchronously open object (object '..."


In [8]:
valid = results_df[np.isfinite(results_df.get("period_difference_pct", np.nan))] if len(results_df) else pd.DataFrame()
if len(valid):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(valid["period_difference_pct"].dropna(), bins=15)
    ax.set_xlabel("Verschil simulatieperiode t.o.v. experiment [%]")
    ax.set_ylabel("Aantal experimenten")
    ax.set_title(f"Periodeverschil zonder demping - {DOF}")
    fig.tight_layout()
    hist_path = PLOTS_DIR / f"{DOF}_period_difference_histogram_no_damping.png"
    fig.savefig(hist_path, dpi=200)
    plt.show()
    print(f"Histogram opgeslagen naar: {hist_path}")
else:
    print("Geen geldige simulatieperiodeverschillen beschikbaar voor histogram.")

Geen geldige simulatieperiodeverschillen beschikbaar voor histogram.


## Checks als de simulatie niet runt

- Pas `MODEL_BY_CONSTRUCTION` aan naar je echte `.dat`/`.sim` modelpaden.
- Controleer `FLOATER_OBJECT_NAME`.
- Controleer de damping-propertynamen in `set_no_damping()`.
- Voor pitch wordt `InitialTrim` gezet, voor roll `InitialHeel`, voor heave `InitialZ`.
- Verlaag `MIN_ABS_INITIAL_AMPLITUDE` als A1/A2 niet gevonden worden.